# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic information from metadata
print(f"Dataset name: {metadata.name}\n\n{metadata.description}\n")

## 2. Data Overview
Review available record sets and fields in the dataset (referencing their `@id` fields).

In [ ]:
# Get the list of available record sets by their @id
if hasattr(metadata, 'record_set') and metadata.record_set:
    record_sets = metadata.record_set
    print(f"Record sets available (by @id):")
    for rs in record_sets:
        print(f"  - {rs['@id']}: {rs.get('name', rs['@id'])}")
else:
    # Use dataset.record_sets property
    record_sets_info = list(dataset.record_sets())
    record_set_ids = [rs['@id'] for rs in record_sets_info]
    print("Record sets found in the schema (by @id):")
    for rs in record_sets_info:
        print(f"  - {rs['@id']}: {rs.get('name', rs['@id'])}")
    record_sets = record_set_ids

# Inspect fields (columns) for each record set
print("\nFields in each record set with their @id:")
for rs_id in record_sets:
    print(f"\nRecord set: {rs_id}")
    try:
        record_set = dataset.record_set(rs_id)
        if 'field' in record_set:
            for field in record_set['field']:
                print(f"  - {field['@id']}: {field.get('name', field['@id'])}")
    except Exception as e:
        print(f"  [Could not extract fields: {e}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this example, fetch record sets (by @id) dynamically
record_sets = list(dataset.record_sets())
record_set_ids = [rs['@id'] for rs in record_sets]

# Store dataframes per record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"Failed to load {record_set_id}: {e}")

# Choose the first record set for demonstration if present
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nSample of the main record set ({main_record_set_id}):")
    display(dataframes[main_record_set_id].head())
else:
    print("No data record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes using fields' `@id`.

In [ ]:
# Identify a numeric field for analysis
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Try to choose a plausible numeric field by column name or by inspection
    numeric_field_candidates = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
    if not numeric_field_candidates:
        # Try to cast numerics if not already
        numeric_field_candidates = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().sum() > 0]
    
    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else None
    print(f"Numeric field selected for EDA: {numeric_field}")

    if numeric_field:
        # Ensure numeric
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        # Filtering
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (shape: {filtered_df.shape}):\n", filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a key attribute (categorical field)
        # Pick the first non-numeric column
        group_field_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        group_field = group_field_candidates[0] if group_field_candidates else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped (mean) {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("Dataset not loaded or no main record set available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If grouped data available, show barplot
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df.sort_values(numeric_field, ascending=False).head(10), x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (Top 10)")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization cannot be produced: necessary field(s) missing.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we explored the FAIR^2 mass spectrometry analysis dataset of extracellular vesicles from stimulated human mast cells.
- We loaded Croissant-compliant metadata and records using `mlcroissant` referencing all dataset elements by `@id`.
- We reviewed available record sets and fields, extracted data to Pandas DataFrames, then identified and analyzed sample numeric and categorical fields.
- We demonstrated filtering, normalization, and grouping, and visualized the distributions in the main record set.

For deeper analysis, consult the official FAIR^2 data documentation and the Croissant schema fields.